# Introduction to Artificial Intelligence - Homework Assignment 06 (5 pts.)
- Name:
- NETID:

**Teaching the Computer to Look**
Due November 2, 2026.

This assignment covers:

- Turning images into tensors, batches, and flattened feature vectors
- Implementing an affine (linear) layer and choosing an initial weight scale
- Why neural networks need nonlinear activations such as ReLU
- Turning logits into probabilities and a numerically stable cross-entropy loss
- Assembling a feed-forward network and checking it against PyTorch

| Task | Description | Points |
|---|---|---:|
| 00 | Load the character evidence | 0.25 |
| 01 | Turn images into tensors and batches | 0.75 |
| 02 | Build an affine layer; explain the initialization | 0.875 |
| 03 | Add a nonlinearity; explain the collapse | 0.375 |
| 04 | Turn logits into a stable loss; interpret the chance loss | 1.125 |
| 05 | Assemble the network | 0.50 |
| 06 | Check against PyTorch and inspect the predictions | 0.75 |
| 07 | Report the results to Caulfield | 0.375 |
| 08 | Export the notebook | 0 |
| | **Total** | **5.00** |

Run the cells in order and answer the written questions. This assignment uses PyTorch but does **not** train anything: every network you build this week keeps its random starting weights. In HW07, we will teach it.

### *Story Progression*

Caulfield finally has readable pages, and your SVM tells him where the cut-out letters sit. That leaves a few hundred characters that still need to be read. The intern volunteered to read them all by hand, then asked whether the squiggly one was a "2" or a "Z". Caulfield would like a second opinion that doesn't drink his coffee.

The forensics lab keeps a reference archive of **9,400 handwritten characters**, each already labeled by an analyst. There are 47 possible labels: ten digits, all 26 uppercase letters, and the 11 lowercase letters that don't look like their uppercase versions. If we can build a model that learns from this archive, we can point it at the recovered evidence.

This week we will build the **forward pass** of a feed-forward neural network: the calculation that turns a 28 × 28 image into one score per character. HW02 gave us a linear model for one number. HW03 gave us a linear model for two classes. Now we will stack several linear layers, add nonlinearities between them, and produce 47 class scores at once.

Our network will start with random weights, so its first guesses will be terrible. That is expected. This week we make sure the calculation is correct; in HW07, we make it accurate.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/lab03/emnist.png" class="img-responsive" alt="Evidence 1: A sample of the forensics lab's reference archive of handwritten characters." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 1: A sample of the forensics lab's reference archive of handwritten characters.</b></figcaption>
</div>

## Task 00: Load the Character Evidence

### Task 00-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_00-1_the_archive.png" alt="What is in the archive: a balanced subset of EMNIST, each example one 28 by 28 grayscale image. Pixels are uint8 values from 0, black background, to 255, bright ink. Labels are class IDs 0 to 46: 0 to 9 are the digits, 10 to 35 are A to Z, 36 to 46 are the lowercase letters a b d e f g h n q r t; ID 10 means A and ID 36 means a. The balanced split merges lowercase letters that look like their capitals, so a handwritten c is labeled C. Files: the training file with X (9400, 28, 28) and y (9400,), the test file with 2,350 held-out images kept closed until HW07, and classes.json with the 47 class names. Do not transpose or rotate the images." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

The setup code loads the arrays for you. Complete one line:

1. Count how many examples belong to each class with `np.bincount`. Pass `minlength=len(classes)` so a missing class would still appear as a zero.

The supplied code prints the counts, the first few labels, and a grid with one example of each class. A **balanced** dataset should have the same count for every class. Look closely at the printed labels: the file stores all 200 examples of one class, then all 200 of the next. That ordering will matter in Task 01.

#### Python Help

```python
labels = np.array([0, 2, 2, 1, 2])
np.bincount(labels)               # array([1, 1, 3]): one 0, one 1, three 2s
np.bincount(labels, minlength=5)  # array([1, 1, 3, 0, 0])
```

#### Useful Links

* [EMNIST dataset description (NIST)](https://www.nist.gov/itl/products-and-services/emnist-dataset)
* [`np.bincount` documentation](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)


### Task 00-1: Code (0.25 pts.)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/wtheisen/nd-cse-30124-homeworks.git'

try:
    import google.colab
except ImportError:
    # Reuse a local checkout, including when launched from a homework subfolder.
    DATA_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / '.git').exists() and (p / 'evidence').is_dir()), None)
    if DATA_ROOT is None:
        DATA_ROOT = Path.cwd() / 'nd-cse-30124-homeworks'
else:
    # Leave the old checkout before deleting it, including on a setup-cell rerun.
    os.chdir('/content')
    DATA_ROOT = Path('/content/nd-cse-30124-homeworks')
    # Fetch fresh evidence every time this cell runs in Colab.
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

if not DATA_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(DATA_ROOT)], check=True)

if not (DATA_ROOT / '.git').exists():
    raise RuntimeError(f'{DATA_ROOT} exists but is not a Git checkout. Choose a different clone location.')

# All later relative paths now start at the homework repository root.
os.chdir(DATA_ROOT)
DATA_ROOT = Path.cwd()

import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

EVIDENCE_DIR = DATA_ROOT / 'evidence' / 'homework06'
required = ['emnist_balanced_small/emnist_balanced_small_train.npz', 'emnist_balanced_small/classes.json']
if any(not (EVIDENCE_DIR / name).is_file() for name in required):
    raise FileNotFoundError('HW06 evidence is missing. Rerun the setup cell with an internet connection, or check that your local repository contains the evidence/homework06 files.')
os.chdir(EVIDENCE_DIR)

raw = np.load('emnist_balanced_small/emnist_balanced_small_train.npz')
X_raw, y_raw = raw['X'], raw['y']
classes = json.loads(Path('emnist_balanced_small/classes.json').read_text())['classes']

# TODO: Count the examples in each class, including classes with zero examples.
counts = None

print('Images:', X_raw.shape, X_raw.dtype, '| pixel range:', X_raw.min(), 'to', X_raw.max())
print('Labels:', y_raw.shape, y_raw.dtype)
print('Classes:', len(classes), '->', ' '.join(classes))
print('Examples per class: min', counts.min(), '| max', counts.max())
print('First 8 labels:', y_raw[:8], '| labels 198-205:', y_raw[198:206])

# Supplied helper: draw a grid of labeled characters.
def show_characters(images, titles, columns=12, suptitle=None):
    rows = math.ceil(len(images) / columns)
    title_lines = max(str(title).count('\n') + 1 for title in titles)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 1.1, rows * (1.1 + 0.3 * title_lines)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, image, title in zip(axes.flat, images, titles):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255 if image.max() > 1 else 1)
        ax.set_title(title, fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

# Supplied: one example of every class, in class-ID order.
first_of_each = [np.flatnonzero(y_raw == label)[0] for label in range(len(classes))]
show_characters(X_raw[first_of_each], [f'{classes[label]} ({label})' for label in range(len(classes))],
                suptitle='One archive example per class: character (class ID)')

### Task 00-1: Expected Output (0 pts.)

```text
Images: (9400, 28, 28) uint8 | pixel range: 0 to 255
Labels: (9400,) int64
Classes: 47 -> 0 1 2 3 4 5 6 7 8 9 A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b d e f g h n q r t
Examples per class: min 200 | max 200
First 8 labels: [45 45 45 45 45 45 45 45] | labels 198-205: [45 45 36 36 36 36 36 36]
```

![One archive example per class](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/expected_samples.png)

*One archive example per class, labeled with its character and class ID. Compare the plot produced by your code with this reference image.*

## Task 01: Turn Images into Tensors and Batches

### Task 01-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_01-1a_arrays_to_tensors.png" alt="From NumPy arrays to tensors. A tensor is like a NumPy array that can also run on a GPU and, in HW07, track gradients. Three fixes: the data type, uint8 to float32, because uint8 arithmetic wraps around; the scale, divide by 255 to put pixels in 0 to 1; the shape, (9400, 28, 28) to (9400, 1, 28, 28) for batch, channels, height, width. Labels stay integer class IDs as torch.long. torch.from_numpy shares memory, so call .copy() first." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_01-1b_order_matters.png" alt="Worked example: the uint8 pixels [[200, 100], [0, 255]] added to themselves give [[144, 200], [0, 254]], because 200 + 200 = 400 wraps to 144. Converting to float and then dividing by 255 once gives [[0.7843, 0.3922], [0, 1]]; unsqueeze(0) adds a dimension of size 1, shape [1, 2, 2]. Dividing twice makes the brightest pixel 1/255 = 0.0039 with no error message." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `to_image_tensor(X)`:

- **Input:** a `uint8` NumPy array with shape `(n, 28, 28)`.
- **Return:** a `torch.float32` tensor with shape `(n, 1, 28, 28)` and values in `[0, 1]`.

Then convert `y_raw` into a `torch.long` tensor named `labels`.

`torch.from_numpy` shares memory with the NumPy array. We call `.copy()` first so later changes to one can't silently change the other.

#### Python Help

```python
a = np.zeros((3, 4), dtype=np.uint8)
torch.from_numpy(a.copy()).float().dtype   # torch.float32
torch.from_numpy(a.copy()).unsqueeze(1).shape   # torch.Size([3, 1, 4])
torch.from_numpy(np.array([3, 1])).long().dtype  # torch.int64 (also called torch.long)
```

#### Useful Links

* [PyTorch tensor tutorial](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
* [`torch.from_numpy`](https://pytorch.org/docs/stable/generated/torch.from_numpy.html) · [`Tensor.unsqueeze`](https://pytorch.org/docs/stable/generated/torch.Tensor.unsqueeze.html)


### Task 01-1: Code (0.25 pts.)

In [ ]:
def to_image_tensor(X):
    # TODO: Convert to float, normalize to [0, 1] exactly once, and add a channel dimension.
    return None

# TODO: Keep the labels as integer class IDs.
labels = None
images = to_image_tensor(X_raw)

print('Images:', tuple(images.shape), images.dtype)
print('Pixel range:', float(images.min()), 'to', float(images.max()))
print('Labels:', tuple(labels.shape), labels.dtype)

### Task 01-1: Expected Output (0 pts.)

```text
Images: (9400, 1, 28, 28) torch.float32
Pixel range: 0.0 to 1.0
Labels: (9400,) torch.int64
```

### Task 01-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_01-2a_split_and_batch.png" alt="Splitting and batching. The file stores all 200 examples of one class, then the next, so the first 940 rows would hold only 5 characters. Shuffle the row numbers once with a fixed seed and split: 940 validation images and 8,460 training images, the same in HW06 and HW07. The test file stays closed until the end of HW07. TensorDataset pairs images with labels; DataLoader returns batches of images (B, 1, 28, 28) and labels (B,). Validation is 7 full batches of 128 and one of 44; training is 67 batches per pass. Shuffling changes the order, never which examples are in each set." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_01-2b_flattening.png" alt="Flattening every dimension except the batch: for two tiny 2 by 2 images holding 0 to 7, batch.flatten() merges both into one vector of 8, while batch.flatten(1) gives one row of 4 per image, shape (2, 4). A real batch goes from (128, 1, 28, 28) to (128, 784). Flattening throws away that pixel 0 and pixel 28 are vertical neighbors; HW07 CNN keeps that structure." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

The split and the `make_loader` helper are supplied.

1. Create `train_loader` (shuffled) and `val_loader` (not shuffled) with `make_loader`. Use the default batch size of 128.
2. Take the first validation batch, `xb, yb = next(iter(val_loader))`. (This line is supplied.)
3. Flatten `xb` into `flat_batch` with shape `(B, 784)`.

#### Python Help

```python
loader = make_loader(some_dataset, shuffle=True)
for xb, yb in loader:     # one batch per loop iteration
    ...
xb, yb = next(iter(loader))   # just the first batch
```

#### Useful Links

* [PyTorch Datasets & DataLoaders tutorial](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
* [`torch.flatten`](https://pytorch.org/docs/stable/generated/torch.flatten.html)


### Task 01-2: Code (0.5 pts.)

In [ ]:
# Supplied: keep the same split across HW06 and HW07. Test data are opened only in HW07.
order = torch.randperm(len(images), generator=torch.Generator().manual_seed(42))
val_idx, train_idx = order[:940], order[940:]
train_data = TensorDataset(images[train_idx], labels[train_idx])
val_data = TensorDataset(images[val_idx], labels[val_idx])

def make_loader(data, shuffle=False, seed=42, batch_size=128):
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)

# TODO: Build a shuffled training loader and an unshuffled validation loader.
train_loader = None
val_loader = None

xb, yb = next(iter(val_loader))

# TODO: Flatten every dimension except the batch dimension.
flat_batch = None

print('Train / validation:', len(train_data), '/', len(val_data))
print('Batches per pass: train', len(train_loader), '| validation', len(val_loader))
print('Last validation batch size:', len(val_data) % 128)
print('Image batch:', tuple(xb.shape), '| label batch:', tuple(yb.shape))
print('Flattened batch:', tuple(flat_batch.shape))
print('Row 0 is still image 0:', torch.equal(flat_batch[0], xb[0].reshape(-1)))

### Task 01-2: Expected Output (0 pts.)

```text
Train / validation: 8460 / 940
Batches per pass: train 67 | validation 8
Last validation batch size: 44
Image batch: (128, 1, 28, 28) | label batch: (128,)
Flattened batch: (128, 784)
Row 0 is still image 0: True
```

![Figure 1. The network we will build this week. The batch dimension B stays separate through every layer; only the feature dimension changes. Numbers in parentheses are each layer's trainable parameters.](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/01_forward_shapes.png)

*Figure 1. The network we will build this week. The batch dimension B stays separate through every layer; only the feature dimension changes. Numbers in parentheses are each layer's trainable parameters.*

## Task 02: Build an Affine Layer

### Task 02-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_02-1a_affine_layer.png" alt="A linear, or affine, layer computes many weighted sums at once: y_j = x dot w_j + b_j for each output unit j, and for a batch Y = X W transpose + b. X is (B, D), W is (H, D) with one row per output unit, b is (H,), Y is (B, H). W is stored outputs by inputs, as in nn.Linear, so we multiply by W transpose: (B, D) times (D, H) gives (B, H). Broadcasting adds the bias row to every example." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_02-1b_worked_example.png" alt="Worked example with W = [[1, 2], [-1, 1], [0, 3]], b = [0, 1, -1] and x = [1, 2]: unit 0 gives 1 + 4 + 0 = 5, unit 1 gives -1 + 2 + 1 = 2, unit 2 gives 0 + 6 - 1 = 5, so y = [5, 2, 5]. The second example [3, 4] gives [11, 2, 11], and X @ W.T + b produces both rows at once. forward also stores its input as self.X, because the HW07 gradient with respect to W needs it." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `LinearLayer.forward(X)`:

- **Input:** `X` with shape `(B, D)`.
- **Return:** `X @ W.T + b` with shape `(B, H)`.
- Store the input in `self.X` before returning.

Use ordinary tensor operations: no `nn.Linear` and no automatic differentiation. The constructor is supplied for now; Task 02-2 explains its weight scale.

#### Python Help

```python
A = torch.tensor([[1., 2.], [3., 4.]])   # (2, 2)
B = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])   # (3, 2)
A @ B.T                                  # (2, 3): row i of A dotted with row j of B
A @ B.T + torch.tensor([10., 20., 30.])  # the bias is added to every row
```

#### Useful Links

* [`nn.Linear` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html): note the shape of its `weight`.


### Task 02-1: Code (0.5 pts.)

In [ ]:
class LinearLayer:
    def __init__(self, input_dim, output_dim, device='cpu'):
        self.W = torch.randn(output_dim, input_dim, device=device) * math.sqrt(2 / input_dim)
        self.b = torch.zeros(output_dim, device=device)

    def forward(self, X):
        # TODO: Cache the input for HW07, then apply the affine transformation.
        self.X = None
        return None

layer = LinearLayer(2, 3)
layer.W = torch.tensor([[1., 2.], [-1., 1.], [0., 3.]])
layer.b = torch.tensor([0., 1., -1.])
X_toy = torch.tensor([[1., 2.], [3., 4.]])

print(layer.forward(X_toy))
print('Output shape:', tuple(layer.forward(X_toy).shape))

first_layer = LinearLayer(784, 256)
print('First real layer:', tuple(flat_batch.shape), '->', tuple(first_layer.forward(flat_batch).shape))

### Task 02-1: Expected Output (0 pts.)

```text
tensor([[ 5.,  2.,  5.],
        [11.,  2., 11.]])
Output shape: (2, 3)
First real layer: (128, 784) -> (128, 256)
```

### Task 02-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_02-2a_breaking_symmetry.png" alt="Where the weights should start: if every weight in a layer starts equal, every unit computes the same output and in HW07 gets the same update, so random weights break the symmetry. Weights too large make sums grow layer after layer; too small, they shrink toward zero. He initialization for layers followed by ReLU draws W_jk from a normal distribution with mean 0 and variance 2/D, with b_j = 0: torch.randn times the square root of 2/D, about 0.051 for D = 784. The factor of 2 compensates for ReLU zeroing about half its inputs." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_02-2b_how_big.png" alt="How big a sum gets: with D = 784 inputs of about 0.5 and weights of standard deviation sigma, one output has standard deviation about the square root of D times sigma times 0.5. With sigma = 1 that is 14; with the He value of about 0.05 it is about 0.7. Three layers with sigma = 1 compound the effect, roughly a factor of 10 per layer for our widths." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

1. Complete the function `he_std(D)`: return the weight standard deviation that `LinearLayer` uses for an input width `D`. (Read it off the supplied constructor in Task 02-1.)
2. Run the supplied experiment. It pushes the real validation batch through three `ReLU(linear)` layers of width 256 using three weight scales and prints the standard deviation of the activations after each layer. It also builds a layer whose weights are all zero and checks whether its four units produce different outputs.

Compare the three rows, then answer Task 02-3.

#### Useful Links

* [He et al. (2015), *Delving Deep into Rectifiers*](https://arxiv.org/abs/1502.01852), Section 2.2
* [`torch.randn`](https://pytorch.org/docs/stable/generated/torch.randn.html)


### Task 02-2: Code (0.25 pts.)

In [ ]:
def he_std(D):
    # TODO: Return the weight standard deviation LinearLayer uses for an input width D.
    return None

# Supplied: the weight standard deviation for each of the three starting rules.
def weight_std(rule, D):
    if rule == 'too small (0.01)':
        return 0.01
    if rule == 'too large (1.0)':
        return 1.0
    return he_std(D)

def activation_scales(X, rule, widths=(256, 256, 256), seed=0):
    generator = torch.Generator().manual_seed(seed)
    stds, H = [], X
    for width in widths:
        W = torch.randn(width, H.shape[1], generator=generator) * weight_std(rule, H.shape[1])
        H = (H @ W.T).clamp_min(0)
        stds.append(round(float(H.std()), 3))
    return stds

for rule in ['too small (0.01)', 'He: sqrt(2 / D)', 'too large (1.0)']:
    print(f'{rule:>17}: activation std after layers 1, 2, 3 = {activation_scales(flat_batch, rule)}')

zero_layer = LinearLayer(784, 4)
zero_layer.W = torch.zeros(4, 784)
print('Zero-initialized units produce identical outputs:',
      bool((zero_layer.forward(flat_batch) == zero_layer.forward(flat_batch)[:, :1]).all()))

### Task 02-2: Reference Output (0 pts.)

```text
 too small (0.01): activation std after layers 1, 2, 3 = [0.061, 0.006, 0.001]
  He: sqrt(2 / D): activation std after layers 1, 2, 3 = [0.308, 0.277, 0.25]
  too large (1.0): activation std after layers 1, 2, 3 = [6.09, 61.951, 633.286]
Zero-initialized units produce identical outputs: True
```

### Task 02-3: Written Response (0.125 pts.)

1. Using your printed standard deviations, describe what happens to the activations as they pass through three layers with each weight scale. Why might the "too large" row cause trouble for the final scores, and why might the "too small" row?
    * `[ANSWER]`
2. The HW03 perceptron started at zero weights and learned perfectly well. Why does a zero start fail for a layer with 256 hidden units?
    * `[ANSWER]`

## Task 03: Add a Nonlinearity

### Task 03-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_03-1a_stacking_collapses.png" alt="Two stacked linear layers are one linear layer: (X W1 transpose + b1) W2 transpose + b2 = X (W2 W1) transpose + (b1 W2 transpose + b2), one matrix and one bias. With single numbers W1 = 2, b1 = 1, W2 = 3, b2 = -1, the result is 6x + 2, still a straight line; with a ReLU in between the line bends at x = -0.5. Neural networks insert a nonlinear activation between the linear layers." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_03-1b_relu.png" alt="ReLU, the rectified linear unit: h = max(0, z), applied to each entry separately; its slope is 1 for positive inputs and 0 for negative ones. Worked example: z = [[-2, 0, 3], [1.5, -0.1, -7]] becomes h = [[0, 0, 3], [1.5, 0, 0]]. The shape never changes, and ReLU stores its input because the HW07 backward pass needs to know which entries were positive." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `ReLU.forward(X)`:

- Store the input in `self.X`.
- Return a tensor of the same shape, with negative entries replaced by zero.

The supplied code then checks the algebra above. It builds two small random linear layers, merges them into one layer, and compares the outputs with and without a ReLU in between.

#### Python Help

```python
torch.tensor([-1., 2.]).clamp_min(0)          # tensor([0., 2.])
torch.maximum(torch.tensor([-1., 2.]), torch.tensor(0.))   # same result
torch.allclose(a, b, atol=1e-6)               # True when every entry is (nearly) equal
```


### Task 03-1: Code (0.25 pts.)

In [ ]:
class ReLU:
    def forward(self, X):
        # TODO: Cache the input for HW07, then keep only its positive values.
        self.X = None
        return None

print(ReLU().forward(torch.tensor([[-2.0, 0.0, 3.0], [1.5, -0.1, -7.0]])))

# Supplied: two stacked linear layers versus one merged layer.
torch.manual_seed(0)
first, second = LinearLayer(4, 5), LinearLayer(5, 3)
first.b, second.b = torch.randn(5), torch.randn(3)
merged = LinearLayer(4, 3)
merged.W = second.W @ first.W
merged.b = first.b @ second.W.T + second.b
X_small = torch.randn(6, 4)

stacked = second.forward(first.forward(X_small))
with_relu = second.forward(ReLU().forward(first.forward(X_small)))
print('Two linear layers equal one merged layer:', torch.allclose(stacked, merged.forward(X_small), atol=1e-5))
print('With ReLU in between, still equal:      ', torch.allclose(with_relu, merged.forward(X_small), atol=1e-5))

### Task 03-1: Expected Output (0 pts.)

```text
tensor([[0.0000, 0.0000, 3.0000],
        [1.5000, 0.0000, 0.0000]])
Two linear layers equal one merged layer: True
With ReLU in between, still equal:       False
```

### Task 03-2: Written Response (0.125 pts.)

Our network will be `Linear → ReLU → Linear → ReLU → Linear`. Why is there **no** ReLU after the final layer? (Hint: think about what a negative final score should be allowed to mean for one of the 47 characters.) In one sentence, compare the role of ReLU here with the role of the RBF kernel in HW04.

* `[ANSWER]`

### *Story Progression*

The intern has been watching over your shoulder. "So it's just multiplying and adding, then throwing away the negatives? I could do that with a calculator." Caulfield points out that a single image needs about 240,000 multiplications. The intern goes to find a bigger calculator.

The network can now produce 47 numbers for each image. Before we assemble it, we need to decide what those numbers mean and how to measure whether they're any good.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework03/intern.png" class="img-responsive" alt="Evidence 2: The intern's attempt to read the letters by hand." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 2: The intern's attempt to read the letters by hand.</b></figcaption>
</div>

## Task 04: Turn Logits into a Stable Loss

### Task 04-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_04-1a_softmax.png" alt="Softmax turns logits into probabilities: p_j = e to the z_j divided by the sum over k of e to the z_k. A logit 1 larger gets about 2.7 times the probability. For logits [2, 1, 0], the exponentials are 7.389, 2.718 and 1.000, total 11.107, and the probabilities 0.665, 0.245 and 0.090. With two classes and logits [s, 0], softmax equals the sigmoid of s: for s = 1.5 both give 0.8176." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_04-1b_overflow.png" alt="Overflow: a 32-bit float tops out near 3.4 times 10 to the 38, and e to the 89 is about 4.49 times 10 to the 38, so exponentiating the logits [1000, 1001, 999] gives inf and then nan. Subtracting the largest logit leaves the probabilities unchanged: [-1, 0, -2] gives p = [0.245, 0.665, 0.090]. In log space, log p_j = z_j minus the log-sum-exp of the logits; for [2, 1, 0], log 11.107 = 2.408, so log p = [-0.408, -1.408, -2.408]." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `log_softmax(logits)` for a `(B, C)` tensor of logits, one row per example:

1. Subtract each row's **maximum** logit from that row (use `keepdim=True` so the shapes broadcast).
2. Compute each row's log-sum-exp of the shifted logits: exponentiate, sum across classes, take the log. Do **not** call `torch.logsumexp` or `torch.log_softmax`; the point is to see how the stable version works.
3. Return the shifted logits minus that row's log-sum-exp.

The supplied code compares your result with PyTorch's `F.log_softmax`, including on logits large enough to overflow a naive softmax.

#### Python Help

```python
Z = torch.tensor([[2., 1., 0.], [5., 7., 6.]])
Z.max(dim=1, keepdim=True).values   # tensor([[2.], [7.]]): shape (2, 1), one max per row
Z.exp().sum(dim=1, keepdim=True)    # one total per row, shape (2, 1)
```

#### Useful Links

* [`torch.Tensor.max`](https://pytorch.org/docs/stable/generated/torch.max.html) · [`F.log_softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)


### Task 04-1: Code (0.5 pts.)

In [ ]:
def log_softmax(logits):
    # TODO: Subtract each row's maximum logit.
    shifted = None
    # TODO: Subtract each row's log-sum-exp from the shifted logits.
    return None

small = torch.tensor([[2., 1., 0.]])
huge = torch.tensor([[1000., 1001., 999.]])
print('Naive softmax of huge logits:', torch.exp(huge) / torch.exp(huge).sum())
print('Log probabilities of [2, 1, 0]:', log_softmax(small).numpy().round(3))
print('Probabilities of [2, 1, 0]:', log_softmax(small).exp().numpy().round(3))
print('Probabilities of [1000, 1001, 999]:', log_softmax(huge).exp().numpy().round(3))

random_logits = torch.randn(5, 47) * 30
print('Each row sums to one:', torch.allclose(log_softmax(random_logits).exp().sum(dim=1), torch.ones(5)))
print('Matches F.log_softmax:', torch.allclose(log_softmax(random_logits), F.log_softmax(random_logits, dim=1), atol=1e-5))

two_class = log_softmax(torch.tensor([[1.5, 0.]])).exp()[0, 0]
print(f'Two-class softmax {float(two_class):.4f} vs sigmoid {float(torch.sigmoid(torch.tensor(1.5))):.4f}')

### Task 04-1: Expected Output (0 pts.)

```text
Naive softmax of huge logits: tensor([[nan, nan, nan]])
Log probabilities of [2, 1, 0]: [[-0.408 -1.408 -2.408]]
Probabilities of [2, 1, 0]: [[0.665 0.245 0.09 ]]
Probabilities of [1000, 1001, 999]: [[0.245 0.665 0.09 ]]
Each row sums to one: True
Matches F.log_softmax: True
Two-class softmax 0.8176 vs sigmoid 0.8176
```

![Figure 2. Softmax turns logits into probabilities. Shifting every logit by the same amount (middle) leaves the probabilities unchanged; multiplying them by a constant (right) makes the prediction more or less confident.](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/02_softmax.png)

*Figure 2. Softmax turns logits into probabilities. Shifting every logit by the same amount (middle) leaves the probabilities unchanged; multiplying them by a constant (right) makes the prediction more or less confident.*

### Task 04-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_04-2a_cross_entropy.png" alt="Cross-entropy for one example with true class y is minus log p_y, which only depends on the probability of the correct class and grows without bound as it falls toward 0. For logits [2, 1, 0]: true class 0 gives a loss of 0.408, true class 2 gives 2.408, with only 9% probability. A batch reports the mean, here (0.408 + 2.408) / 2 = 1.408. With two classes this is the HW03 log loss." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_04-2b_selecting_entries.png" alt="Paired integer indexing selects one entry per row: log_p[torch.arange(2), targets] with targets [0, 2] picks entries [0, 0] and [1, 2], giving [-0.4, -2.4]; log_p[:, targets] instead returns a (2, 2) table of columns 0 and 2 for every row. F.cross_entropy takes raw logits; passing softmax probabilities applies softmax twice and gives 1.0848 instead of 1.4076 for the worked example." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `cross_entropy(logits, targets)`:

- **Input:** `logits` with shape `(B, C)` and `targets` with shape `(B,)` containing integer class IDs.
- **Return:** a scalar tensor: the mean negative log probability of each example's correct class.

Use your `log_softmax`. The supplied code compares your loss with `F.cross_entropy` on the worked example and on the huge logits from Task 04-1.

#### Useful Links

* [`F.cross_entropy` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
* [PyTorch indexing with tensors](https://pytorch.org/docs/stable/tensor_view.html) · [`torch.arange`](https://pytorch.org/docs/stable/generated/torch.arange.html)


### Task 04-2: Code (0.5 pts.)

In [ ]:
def cross_entropy(logits, targets):
    # TODO: Select each example's correct-class log probability and average the negatives.
    log_p = None
    return None

pair = torch.tensor([[2., 1., 0.], [2., 1., 0.]])
pair_targets = torch.tensor([0, 2])
print(f'Worked example loss: ours {float(cross_entropy(pair, pair_targets)):.4f} | '
      f'PyTorch {float(F.cross_entropy(pair, pair_targets)):.4f}')

huge_pair = torch.tensor([[1000., 1001., 999.], [-1000., -1001., -999.]])
huge_targets = torch.tensor([1, 2])
print(f'Huge logits loss:    ours {float(cross_entropy(huge_pair, huge_targets)):.4f} | '
      f'PyTorch {float(F.cross_entropy(huge_pair, huge_targets)):.4f}')

probabilities = F.softmax(pair, dim=1)
print(f'Common bug, softmax passed to F.cross_entropy: {float(F.cross_entropy(probabilities, pair_targets)):.4f}')

uniform = torch.zeros(4, 47)   # every class gets the same logit
print(f'Loss when all 47 classes are equally likely: {float(cross_entropy(uniform, torch.tensor([0, 5, 20, 46]))):.4f}')

### Task 04-2: Expected Output (0 pts.)

```text
Worked example loss: ours 1.4076 | PyTorch 1.4076
Huge logits loss:    ours 0.4076 | PyTorch 0.4076
Common bug, softmax passed to F.cross_entropy: 1.0848
Loss when all 47 classes are equally likely: 3.8501
```

### Task 04-3: Written Response (0.125 pts.)

The last printed line is the loss of a model that assigns every class the same probability. Show where the number 3.8501 comes from. In HW07, a training loss that stays near this value after many updates tells us something specific about the model. What?

* `[ANSWER]`

## Task 05: Assemble the Network

### Task 05-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_05-1a_putting_in_order.png" alt="The multilayer perceptron: 784 pixels, Linear to 256, ReLU, Linear to 128, ReLU, Linear to 47 logits. The widths 256 and 128 are hyperparameters; 784 pixels and 47 classes are not choices, and the last layer has no ReLU. The forward pass flattens the batch and feeds each layer output into the next; rows stay independent, so any batch size runs the same code." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_05-1b_counting_parameters.png" alt="Counting parameters, H times D weights plus H biases per linear layer: 784 to 256 has 200,704 weights and 256 biases, 200,960 in all; 256 to 128 has 32,768 and 128, 32,896; 128 to 47 has 6,016 and 47, 6,063. Total 239,919, about 84% of them in the first layer." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `ScratchFFN.forward(X)`:

- **Input:** an image batch with shape `(B, 1, 28, 28)`.
- **Return:** logits with shape `(B, 47)`.

1. Flatten everything except the batch dimension.
2. Pass the result through each layer in `self.layers`, **in order**, calling each layer's `forward`.

The supplied code traces the shape after every layer, counts the parameters, and evaluates the loss of the untrained network on one batch.

#### Python Help

```python
toy_layers = [LinearLayer(2, 3), ReLU(), LinearLayer(3, 1)]
H = torch.ones(5, 2)            # 5 examples, 2 features each
for layer in toy_layers:
    H = layer.forward(H)        # each layer's output becomes the next layer's input
print(H.shape)                  # torch.Size([5, 1])
```


### Task 05-1: Code (0.5 pts.)

In [ ]:
class ScratchFFN:
    def __init__(self, device='cpu'):
        self.layers = [LinearLayer(784, 256, device), ReLU(),
                       LinearLayer(256, 128, device), ReLU(),
                       LinearLayer(128, 47, device)]

    def forward(self, X):
        # TODO: Flatten each image, then apply every layer in order.
        return None

torch.manual_seed(42)
model = ScratchFFN()
xb4, yb4 = next(iter(make_loader(val_data, batch_size=4)))

activations = xb4.flatten(1)
shapes = [tuple(activations.shape)]
for layer in model.layers:
    activations = layer.forward(activations)
    shapes.append((type(layer).__name__, tuple(activations.shape)))
print('Input:', shapes[0])
for name, shape in shapes[1:]:
    print(f'  after {name:<11} {shape}')

parameter_count = sum(layer.W.numel() + layer.b.numel() for layer in model.layers if hasattr(layer, 'W'))
print('Trainable parameters:', parameter_count)
print('forward() output shape:', tuple(model.forward(xb4).shape))
print(f'Untrained loss on the first validation batch: {float(cross_entropy(model.forward(xb), yb)):.4f}')

### Task 05-1: Expected Output (0 pts.)

The architecture lines are exact. The loss depends on the random starting weights; yours should match when you run the cells in order.

```text
Input: (4, 784)
  after LinearLayer (4, 256)
  after ReLU        (4, 256)
  after LinearLayer (4, 128)
  after ReLU        (4, 128)
  after LinearLayer (4, 47)
Trainable parameters: 239919
forward() output shape: (4, 47)
Untrained loss on the first validation batch: 3.9775
```

### *Story Progression*

The network is assembled: 239,919 numbers, every one of them chosen at random. Caulfield wants to know whether it works. The honest answer is that we don't know yet whether it *computes* what we think it computes, never mind whether it reads handwriting.

Those are two different questions, and the next task separates them. First we'll check our scratch implementation against PyTorch's. Then we'll see what an untrained network actually predicts.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/lab03/deeper.png" class="img-responsive" alt="Evidence 3: The evidence board for our three-layer network. Every one of those strings is a weight." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 3: The evidence board for our three-layer network. Every one of those strings is a weight.</b></figcaption>
</div>

## Task 06: Check Against PyTorch and Inspect the Predictions

### Task 06-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_06-1a_library_way.png" alt="The library versions of our pieces: X.flatten(1) is nn.Flatten(), LinearLayer(D, H) is nn.Linear(D, H) with weight (H, D) and bias (H,), ReLU() is nn.ReLU(), a list of layers with a loop is nn.Sequential, and cross_entropy is F.cross_entropy on raw logits. A PyTorch model inherits from nn.Module, builds its layers in __init__, defines forward, and is called like a function." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_06-1b_same_function.png" alt="Same architecture is not the same function: nn.Linear draws its own random weights, so the two models only agree after every scratch W and b is copied into the paired PyTorch .weight and .bias with .copy_(), in order. Then both get the same images and their logits should agree to within rounding. Do the copy inside torch.no_grad()." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

1. Complete `TorchFFN.__init__`: build `self.layers` with `nn.Sequential`, matching the scratch architecture exactly: `Flatten`, three `Linear` layers, and a `ReLU` after each of the first two.
2. Complete `copy_to_torch(scratch, reference)`. The supplied code pairs the scratch linear layers with the PyTorch linear layers in order. For each pair, copy `W` into `.weight` and `b` into `.bias` with `.copy_()`.

The supplied code prints the largest difference between the two models' logits.

#### Python Help

```python
net = nn.Sequential(nn.Linear(3, 2), nn.ReLU())
net[0].weight.shape                 # torch.Size([2, 3])
with torch.no_grad():
    net[0].weight.copy_(torch.ones(2, 3))   # overwrite in place
```

#### Useful Links

* [Build the neural network (PyTorch tutorial)](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
* [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) · [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) · [`torch.no_grad`](https://pytorch.org/docs/stable/generated/torch.no_grad.html)


### Task 06-1: Code (0.5 pts.)

In [ ]:
class TorchFFN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Match the scratch architecture exactly.
        self.layers = None

    def forward(self, X):
        return self.layers(X)

def copy_to_torch(scratch, reference):
    scratch_linear = [layer for layer in scratch.layers if isinstance(layer, LinearLayer)]
    torch_linear = [layer for layer in reference.layers if isinstance(layer, nn.Linear)]
    with torch.no_grad():
        for ours, theirs in zip(scratch_linear, torch_linear):
            # TODO: Copy this layer's weights and bias into the PyTorch layer.
            pass

reference = TorchFFN()
print(reference)

with torch.no_grad():
    before = (model.forward(xb) - reference(xb)).abs().max()
    copy_to_torch(model, reference)
    after = (model.forward(xb) - reference(xb)).abs().max()
print(f'Largest logit difference before copying weights: {float(before):.2f}')
print('Largest logit difference after copying is below 1e-5:', bool(after < 1e-5))

### Task 06-1: Expected Output (0 pts.)

The difference before copying depends on PyTorch's random initialization and will vary; it should be well above zero. The other lines are exact.

```text
TorchFFN(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=47, bias=True)
  )
)
Largest logit difference before copying weights: 2.04
Largest logit difference after copying is below 1e-5: True
```

### Task 06-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_06-2a_evaluating.png" alt="Evaluating on the whole validation set: the last batch is shorter, so add up the total loss and the number correct over every batch and divide by the total number of examples once. Worked example: 3 of 4 correct with total loss 2.0, then 0 of 1 with loss 4.0, gives accuracy 3/5 = 0.60 and loss 6.0/5 = 1.2; averaging batch accuracies would give 0.375. Use reduction sum, and predict with argmax of the logits." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/plates/plate_06-2b_what_to_expect.png" alt="What to expect: uniform guessing among 47 classes gives accuracy 1/47, about 2.1%, and loss log 47, about 3.85. A particular random network will not hit those exactly, but its confidence, the largest softmax probability, should be low." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete the loop inside `evaluate(model, loader)` so it accumulates `total_loss`, `correct`, and `count` across batches. It works for both the scratch model and the PyTorch model.

#### Python Help

```python
logits = torch.tensor([[0.1, 2.0], [3.0, -1.0], [0.0, 0.5]])
targets = torch.tensor([1, 1, 1])
logits.argmax(dim=1)                                  # tensor([1, 0, 1])
int((logits.argmax(dim=1) == targets).sum())          # 2 correct
float(F.cross_entropy(logits, targets, reduction='sum'))   # total, not mean
```


### Task 06-2: Code (0.25 pts.)

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    total_loss, correct, count = 0., 0, 0
    for xb_eval, yb_eval in loader:
        logits = model(xb_eval) if isinstance(model, nn.Module) else model.forward(xb_eval)
        # TODO: Accumulate the summed loss, number correct, and number of examples.
        total_loss += 0
        correct += 0
        count += 0
    return {'loss': round(total_loss / count, 4), 'accuracy': round(correct / count, 4)}

print('Untrained scratch FFN, validation:', evaluate(model, val_loader))
print('Untrained PyTorch FFN, validation:', evaluate(reference, val_loader))
print(f'Uniform guessing: loss {math.log(47):.4f}, accuracy {1 / 47:.4f}')

with torch.no_grad():
    probabilities = log_softmax(model.forward(xb)).exp()
    confidence, predicted = probabilities.max(dim=1)
print(f'Average confidence in its own prediction: {float(confidence.mean()):.3f}')
print('Distinct classes predicted for 128 images:', len(predicted.unique()))

# Supplied: show the first 12 validation images with the untrained model's guesses.
show_characters(xb[:12, 0].numpy(),
                [f'true {classes[int(t)]}\nguess {classes[int(p)]} ({float(c):.2f})'
                 for t, p, c in zip(yb[:12], predicted[:12], confidence[:12])],
                columns=6, suptitle='The untrained network\'s guesses (and confidence)')

### Task 06-2: Reference Output (0 pts.)

Your numbers should match when you run the cells in order. Small floating-point differences across machines are normal.

```text
Untrained scratch FFN, validation: {'loss': 3.9704, 'accuracy': 0.0234}
Untrained PyTorch FFN, validation: {'loss': 3.9704, 'accuracy': 0.0234}
Uniform guessing: loss 3.8501, accuracy 0.0213
Average confidence in its own prediction: 0.056
Distinct classes predicted for 128 images: 17
```

![The untrained network's guesses](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/reference_untrained_guesses.png)

*The untrained network's guesses and confidence on the first 12 validation images.*

## Task 07: Report the Results to Caulfield
### Task 07-1: Written Response (0.375 pts.)

1. Our scratch network and PyTorch's network agreed to within $10^{-5}$, yet both get only a few percent of the validation set right. Explain why both statements can be true. What did the comparison verify, and what did it not verify?
    * `[ANSWER]`
2. Report the untrained validation loss and accuracy. Compare them with uniform guessing ($\log 47$ and $1/47$). Why isn't a random network exactly at chance?
    * `[ANSWER]`
3. Before computing the loss, we keep the network's output as logits rather than converting it to probabilities. Give two reasons, using what you saw in Task 04.
    * `[ANSWER]`

### *Story Progression*

Your report goes to Caulfield: the network computes exactly what it's supposed to, and it reads handwriting about as well as a person picking characters out of a hat. The intern feels much better about his own accuracy.

Caulfield isn't discouraged. A calculation we can trust is the foundation; what's missing is **learning**. Every one of those 239,919 weights needs to be nudged, a little at a time, in the direction that lowers the loss you just implemented. That requires knowing how the loss changes when each weight changes.

In HW07, we'll implement the backward pass, train the network on the archive, compare it with PyTorch's trained version and a convolutional network, and finally point it at the recovered letters.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework06/evidence_board.png" class="img-responsive" alt="Evidence 4: Caulfield's evidence board: the recovered cut-out pages, the letters we still can't read, and the untrained network's results." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 4: Caulfield's evidence board: the recovered cut-out pages, the letters we still can't read, and the untrained network's results.</b></figcaption>
</div>

## Task 08: Export the Completed Notebook (0 pts.)

1. Complete your code and written responses, and run the cells above in order. Compare your results with the expected and reference outputs.
2. **Save your notebook**, including its outputs. Set `GENERATE = True` below when you are ready to export.
3. In Colab, the cell captures the current notebook and its outputs. Locally, it reads the saved notebook at `NOTEBOOK_PATH`; adjust that path if you saved your copy elsewhere.
4. Download **`homework06.html`** and open it in a browser. Check that your name, NETID, code, written answers, and outputs appear.

Submit only **`homework06.html`** on Canvas. The exporter does not rerun your code.

### Task 08-1: Code (0 pts.)

In [ ]:
GENERATE = False
NOTEBOOK_PATH = DATA_ROOT / 'homeworks' / 'homework06' / 'homework06.ipynb'

if GENERATE:
    import json
    import nbformat
    from nbconvert import HTMLExporter
    from pathlib import Path

    try:
        from google.colab import _message, files
    except ImportError:
        current = nbformat.read(NOTEBOOK_PATH, as_version=4)
        html_path = NOTEBOOK_PATH.with_name('homework06.html')
        in_colab = False
    else:
        captured = _message.blocking_request('get_ipynb', timeout_sec=10)
        current = nbformat.reads(json.dumps(captured['ipynb']), as_version=4)
        html_path = Path('/content/homework06.html')
        in_colab = True

    exporter = HTMLExporter()
    exporter.embed_images = True
    report_html, _ = exporter.from_notebook_node(current)
    html_path.write_text(report_html, encoding='utf-8')
    print(f'Exported {html_path.name}. Open it in a browser before submitting.')

    if in_colab:
        files.download(str(html_path))
    else:
        print(f'Saved to: {html_path}')
else:
    print('Export disabled. Set GENERATE = True when ready.')

### Task 08-1: Expected Output (0 pts.)

```text
Export disabled. Set GENERATE = True when ready.
```